# Section 4 – Data Preparation and Preprocessing
**Water-related datasets for the Philippines (PSA + OWID/FAO AQUASTAT)**

This notebook follows the *Section 4 Practical Guide* step by step:

| Part | Content |
|---|---|
| 4.1 | Data Selection |
| 4.2 | Data Cleaning |
| 4.3 | Data Transformation |
| 4.4 | Data Integration |
| 4.5 | Prepared Dataset Description and Validation + Export |

**Files needed (upload when prompted in Cell 2, or place them in Google Drive):**
1. `Physical Water Supply.csv` (PSA) – also accepted as `Physical_Water_Supply.csv`
2. `Physical Water Use.csv` (PSA) – also accepted as `Physical_Water_Use.csv`
3. `industrial-water-withdrawal.csv` (OWID / FAO AQUASTAT)
4. The PSA Water Accounts Excel workbook (`FINAL - 26ENREAS01-114 - 2025 Water Accounts Web Release Materials - Web Release Materials - 4 of 4.xlsx`)

**Ground rules (from the guide):** original files are never overwritten; all work is on copies; missing values are *not* filled with zero; zeros and outliers are investigated, not deleted; OWID and PSA indicators are kept distinct (different definitions); correlations do not prove causation.

## Step 0 – Setup: imports, file locations, and a cleaning/transformation log

In [ ]:
import os, glob, re, shutil, sys, warnings
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---- Where are the input files? -------------------------------------------
# Option A (default): upload them in Colab when prompted below.
# Option B: mount Google Drive and point DATA_DIR at the folder that holds them, e.g.
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_DIR = '/content/drive/MyDrive/Water_Data'
DATA_DIR = "."

IN_COLAB = "google.colab" in sys.modules

def _norm(name):
    return re.sub(r"[\s_\-]+", "", name.lower())

def find_file(*candidates, ext=None, contains=None):
    """Find a file in DATA_DIR by (space/underscore/case-insensitive) name, or by extension + keyword."""
    files = [f for f in os.listdir(DATA_DIR) if os.path.isfile(os.path.join(DATA_DIR, f))]
    for cand in candidates:
        for f in files:
            if _norm(f) == _norm(cand):
                return os.path.join(DATA_DIR, f)
    if ext:
        hits = [f for f in files if f.lower().endswith(ext) and (contains is None or contains.lower() in f.lower())]
        if hits:
            return os.path.join(DATA_DIR, sorted(hits)[0])
    return None

def all_inputs_present():
    return all([
        find_file("Physical Water Supply.csv"),
        find_file("Physical Water Use.csv"),
        find_file("industrial-water-withdrawal.csv"),
        find_file(ext=".xlsx", contains="water accounts") or find_file(ext=".xlsx"),
    ])

if not all_inputs_present() and IN_COLAB:
    from google.colab import files
    print("Please upload the 4 input files (3 CSV + 1 Excel workbook).")
    uploaded = files.upload()

print("Supply   :", find_file("Physical Water Supply.csv"))
print("Use      :", find_file("Physical Water Use.csv"))
print("Withdraw :", find_file("industrial-water-withdrawal.csv"))
print("Accounts :", find_file(ext=".xlsx", contains="water accounts") or find_file(ext=".xlsx"))

In [ ]:
# Running logs - the guide asks us to keep a record of every cleaning decision / transformation.
cleaning_log = []
transformation_log = []

def log_clean(dataset, column, n_records, issue, genuine_problem, action):
    cleaning_log.append({
        "dataset": dataset, "column": column, "affected_records": n_records,
        "issue": issue, "genuine_data_quality_problem": genuine_problem, "action_or_reason": action
    })

def log_transform(dataset, transformation, reason):
    transformation_log.append({"dataset": dataset, "transformation": transformation, "reason": reason})

---
# 4.1 DATA SELECTION
## Steps 1–3. Load the original datasets and create working copies
The PSA CSVs are semicolon-separated, have two title rows to skip, and use `.` / `..` as missing-value markers.

In [ ]:
supply_path   = find_file("Physical Water Supply.csv")
use_path      = find_file("Physical Water Use.csv")
withdraw_path = find_file("industrial-water-withdrawal.csv")
accounts_path = find_file(ext=".xlsx", contains="water accounts") or find_file(ext=".xlsx")

if accounts_path is None:
    raise FileNotFoundError(
        "The PSA Water Accounts Excel workbook (.xlsx) was not found in DATA_DIR. "
        "Upload it (Cell 2) before continuing - Worksheets 3 and 4 are required in 4.3."
    )

supply = pd.read_csv(supply_path, sep=";", skiprows=2, na_values=[".", ".."])
use = pd.read_csv(use_path, sep=";", skiprows=2, na_values=[".", ".."])
withdrawal = pd.read_csv(withdraw_path)
water_accounts = pd.ExcelFile(accounts_path)

print("Workbook sheets:", water_accounts.sheet_names)
print("Supply:", supply.shape, "| Use:", use.shape, "| Withdrawal:", withdrawal.shape)

In [ ]:
# Working copies: the originals (supply, use, withdrawal) are never modified again.
supply_work = supply.copy()
use_work = use.copy()
withdrawal_work = withdrawal.copy()

## Step 4. Select the Philippine records from Industrial Water Withdrawal

In [ ]:
philippines = withdrawal_work[withdrawal_work["Code"] == "PHL"].copy()
philippines = philippines.sort_values("Year")

display(philippines.head())
print("Philippine records:", len(philippines))
print("Years:", philippines["Year"].min(), "-", philippines["Year"].max())
assert len(philippines) == 29, "Expected 29 Philippine records (1995-2023)"

## Step 5. Inspect the relevant PSA Water Accounts worksheets (3 and 4)

In [ ]:
for sheet in ["3", "4"]:
    df = pd.read_excel(water_accounts, sheet_name=sheet, header=None)
    print("Worksheet:", sheet)
    print("Dimensions:", df.shape)
    display(df.head(15))

## Step 6. Variable-selection guide

| Dataset | Variables to examine |
|---|---|
| Physical Water Supply | Year, Inputs, Total Supply, sector-related supply columns |
| Physical Water Use | Year, Inputs, Total Use, sector-related use columns |
| Industrial Water Withdrawal | Entity, Code, Year, Industrial water withdrawal |
| Water Use Efficiency | Year, Industry water used, Industry water-use efficiency, Overall water-use efficiency |
| Level of Water Stress | Year, Total Water Withdrawals, Level of Water Stress |

**Important considerations**
- `Year` and `Inputs` are retained (Inputs identifies the accounting category of each row).
- The PSA industry category = **mining + manufacturing + construction** – never describe it as mining-only.
- OWID *Industrial water withdrawal* = **self-supplied industrial withdrawal**, not total industrial consumption.
- Aggregate totals and their components must not be fed together to a mining model as independent observations (double-counting).

## Step 7. Inspect the PSA accounting categories before selecting rows

In [ ]:
print("PHYSICAL WATER SUPPLY CATEGORIES")
for v in supply_work["Inputs"].unique(): print("  ", repr(v))

print("\nPHYSICAL WATER USE CATEGORIES")
for v in use_work["Inputs"].unique(): print("  ", repr(v))

Review of the labels (compare with the PSA publication definitions):

- `..Surface water`, `..Ground water` → components of **TOTAL SUPPLY/USE ABSTRACTED WATER**
- `..For distribution`, `..For own-use` / `..Distributed water`, `..Own use of water` → distribution vs. own-use
- `..To inland water resources`, `..To other sources`, `..Return to ...` → components of **TOTAL RETURN FLOWS**
- `..Evaporation`, `..Transpiration`, `..Water incorporated to products` → components of **TOTAL WATER EVAPORATED, TRANSPIRED AND INCORPORATED INTO PRODUCTS**
- `TOTAL SUPPLY` / `TOTAL USE` → the overall national totals (one row per year)

Rows prefixed with `..` are components; rows in capitals are aggregates. Mixing them in one model would double-count.

## Step 8. Check the selected dataset dimensions and save the Philippine subset

In [ ]:
selected_datasets = {
    "Physical Water Supply": supply_work,
    "Physical Water Use": use_work,
    "Philippine Industrial Water Withdrawal": philippines,
}
for name, df in selected_datasets.items():
    print("\nDataset:", name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print(df.columns.tolist())

philippines.to_csv("selected_philippine_industrial_withdrawal.csv", index=False)

## Step 9. Checkpoint

In [ ]:
assert (philippines["Code"] == "PHL").all(), "Subset contains non-PHL records"
assert supply.equals(supply_work) and use.equals(use_work) and withdrawal.equals(withdrawal_work), "Originals changed!"
print("✔ Philippine subset contains only Code = PHL")
print("✔ Worksheets 3 and 4 selected for the Water Accounts")
print("✔ Original DataFrames unchanged (copies used)")
print("✔ No records deleted: supply", supply.shape, "->", supply_work.shape,
      "| use", use.shape, "->", use_work.shape)
print("   (Only the Philippine subset of the OWID file was taken; other countries remain in `withdrawal`.)")

---
# 4.2 DATA CLEANING
Cleaning decisions are based on the **meaning** of the data, not just on the presence of blanks or zeros.

## Step 1. Inspect missing values

In [ ]:
cleaning_datasets = {
    "Physical Water Supply": supply_work,
    "Physical Water Use": use_work,
    "Philippine Industrial Water Withdrawal": philippines,
}
for name, df in cleaning_datasets.items():
    print("\nDATASET:", name)
    missing = df.isnull().sum()
    summary = pd.DataFrame({"Missing Values": missing, "Percentage": (missing / len(df)) * 100})
    display(summary)

## Step 2. Investigate missing values in Industrial Water Withdrawal

In [ ]:
missing_codes = withdrawal_work[withdrawal_work["Code"].isnull()]
print("Records with missing country codes:")
display(missing_codes.head(20))
print("Total records with missing codes:", len(missing_codes))
print("\nDistinct entities with missing code:", missing_codes["Entity"].nunique())
print(sorted(missing_codes["Entity"].unique()))

print("\nMissing values in Philippine dataset:")
display(philippines.isnull().sum())

In [ ]:
log_clean("Industrial Water Withdrawal (full international file)", "Code", int(len(missing_codes)),
          "Missing country codes; affected entities are FAO regional aggregates (e.g. 'Eastern Asia (FAO)'), not countries.",
          "No - structurally expected (regions have no ISO country code)",
          "Not filled. Not relevant to the PHL subset; regional rows are excluded by the Code == 'PHL' selection.")
log_clean("Philippine Industrial Water Withdrawal", "all columns", int(philippines.isnull().sum().sum()),
          "Checked for missing values in the PHL subset.", "No",
          "No imputation required.")

## Step 3. Investigate missing values in the PSA supply and use tables
The PSA tables are accounting tables: a blank usually means the entry *does not apply* to that sector. These blanks are **not** replaced with zero.

In [ ]:
print("SUPPLY ROWS WITH MISSING VALUES")
display(supply_work[supply_work.isnull().any(axis=1)].head(20))
print("USE ROWS WITH MISSING VALUES")
display(use_work[use_work.isnull().any(axis=1)].head(20))

In [ ]:
# Which accounting rows are blank in which columns? (helps classify structural vs. unreported blanks)
def blanks_by_row(df, label):
    num_cols = df.columns.drop(["Year", "Inputs"])
    out = df.groupby("Inputs", sort=False)[list(num_cols)].apply(lambda g: g.isnull().all())
    print(label, "- True = column is blank for EVERY year in that accounting row")
    display(out.T.rename(columns=lambda c: c[:28]))

blanks_by_row(supply_work, "SUPPLY")
blanks_by_row(use_work, "USE")

**Classification of the PSA blanks**

| Pattern | Classification |
|---|---|
| A column is blank in *every* year for a given accounting row (e.g. `Accumulation`, `Flows from the environment` for rows other than the abstraction rows, `Households` for most rows) | **Structurally not applicable** – the accounting entry has no value for that sector |
| A blank in only some years within a row that otherwise has values | Would be **unreported/unavailable** – check against the PSA publication (the cell above lets you verify this; none should be assumed) |

Because the blanks follow the accounting structure, they are retained as missing (`NaN`) and **not** filled with zero.

In [ ]:
for name, df in [("Physical Water Supply", supply_work), ("Physical Water Use", use_work)]:
    miss = df.isnull().sum()
    for col, n in miss[miss > 0].items():
        log_clean(name, col, int(n),
                  "Blank cells ('.' in source) in accounting table",
                  "No - structurally not applicable to that accounting entry/sector (verify against PSA definitions)",
                  "Retained as NaN; not replaced with zero")

## Step 4. Check duplicate records again

In [ ]:
print("SUPPLY DUPLICATES:", supply_work.duplicated().sum())
print("USE DUPLICATES:", use_work.duplicated().sum())
print("PHILIPPINE WITHDRAWAL DUPLICATES:", philippines.duplicated().sum())

# stricter key-based checks (one row per Year x Inputs / per Year)
print("\nSupply duplicate (Year, Inputs):", supply_work.duplicated(["Year", "Inputs"]).sum())
print("Use duplicate (Year, Inputs):", use_work.duplicated(["Year", "Inputs"]).sum())
print("PHL duplicate Year:", philippines.duplicated(["Year"]).sum())

log_clean("Supply / Use / PHL withdrawal", "all", int(supply_work.duplicated().sum() + use_work.duplicated().sum() + philippines.duplicated().sum()),
          "Exact-duplicate check", "No", "No duplicate removal necessary")

## Step 5. Check data types

In [ ]:
for name, df in cleaning_datasets.items():
    print("\nDATASET:", name)
    print(df.dtypes)

## Step 6. Convert numeric columns when necessary

In [ ]:
print(philippines["Industrial water withdrawal"].dtype)

In [ ]:
# Only convert if the column is NOT already numeric - and check that conversion creates no new missing values.
col = "Industrial water withdrawal"
if not pd.api.types.is_numeric_dtype(philippines[col]):
    before = philippines[col].isnull().sum()
    philippines[col] = pd.to_numeric(philippines[col], errors="coerce")
    after = philippines[col].isnull().sum()
    print("New missing values introduced by conversion:", after - before)
else:
    print("Column is already numeric - no conversion required.")
print("Missing after check:", philippines[col].isnull().sum())

## Step 7. Investigate zero values

In [ ]:
print("SUPPLY ZERO VALUES")
print((supply_work.select_dtypes("number") == 0).sum())
print("\nUSE ZERO VALUES")
print((use_work.select_dtypes("number") == 0).sum())
print("\nZero withdrawal values (PHL):", (philippines["Industrial water withdrawal"] == 0).sum())

In [ ]:
# Show where the zeros occur so they can be judged (true zero vs. 'not applicable' entry).
for name, df in [("SUPPLY", supply_work), ("USE", use_work)]:
    num = df.select_dtypes("number").drop(columns=["Year"])
    zero_rows = df[(num == 0).any(axis=1)]
    print(name, "- rows containing at least one zero:", len(zero_rows))
    display(zero_rows.groupby("Inputs").size().rename("rows_with_zero").to_frame())

# Zero in the *Total* columns (Total Supply / Total Use) would be a genuine concern:
print("Zeros in Total Supply:", (supply_work["Total Supply"] == 0).sum(), "| Zeros in Total Use:", (use_work["Total Use"] == 0).sum())

In [ ]:
for name, df in [("Physical Water Supply", supply_work), ("Physical Water Use", use_work)]:
    zeros = (df.select_dtypes("number").drop(columns=["Year"]) == 0).sum()
    for col, n in zeros[zeros > 0].items():
        log_clean(name, col, int(n), "Zero values in accounting table",
                  "Not determined to be an error - zero may be an actual zero quantity (e.g. 'Water Supply' sector own-use of an item) ",
                  "Retained unchanged; confirm meaning against PSA definitions if the column is used in analysis")
log_clean("Philippine Industrial Water Withdrawal", "Industrial water withdrawal",
          int((philippines["Industrial water withdrawal"] == 0).sum()), "Zero-value check", "No", "No zeros present; nothing to treat")

## Step 8. Check unusual values

In [ ]:
print(supply_work.describe())
print(use_work.describe())
print(philippines.describe())

In [ ]:
# Context for the Philippine series: year-over-year change (large jumps are flagged, NOT removed).
p = philippines.sort_values("Year").copy()
p["yoy_pct_change"] = p["Industrial water withdrawal"].pct_change() * 100
display(p[["Year", "Industrial water withdrawal", "yoy_pct_change"]].round(1))
print("Largest year-over-year changes:")
display(p.reindex(p["yoy_pct_change"].abs().sort_values(ascending=False).index)[["Year", "yoy_pct_change"]].head(5).round(1))

Large year-to-year movements (e.g. the jump into 2017–2018 and the fall in 2019 visible above) are **flagged for verification against the FAO AQUASTAT source, not deleted**. The OWID series for some years may be interpolated by the source; the original values are retained.

In [ ]:
log_clean("Philippine Industrial Water Withdrawal", "Industrial water withdrawal", 0,
          "Large year-on-year changes (see yoy table); possible interpolation in source series",
          "Unverified - needs confirmation against FAO AQUASTAT", "Retained; no outlier removed")

## Step 9. Cleaning decision log

In [ ]:
cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)

---
# 4.3 DATA TRANSFORMATION
## Step 1. Standardize column names

In [ ]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(",", "", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("/", "_")
    )
    return df

supply_clean = standardize_columns(supply_work)
use_clean = standardize_columns(use_work)
withdrawal_clean = standardize_columns(philippines)

print(supply_clean.columns.tolist())
print(use_clean.columns.tolist())
print(withdrawal_clean.columns.tolist())
log_transform("Supply / Use / PHL withdrawal", "Column names -> lowercase_with_underscores", "Consistent, easy-to-use variable names")

## Step 2. Standardize the Year column

In [ ]:
for df in [supply_clean, use_clean, withdrawal_clean]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

print(supply_clean["year"].dtype)
print(use_clean["year"].dtype)
print(withdrawal_clean["year"].dtype)
print("Missing years:", supply_clean["year"].isnull().sum(), use_clean["year"].isnull().sum(), withdrawal_clean["year"].isnull().sum())
log_transform("Supply / Use / PHL withdrawal", "year -> Int64", "Consistent integer key for merging")

## Step 3. Extract the Water Use Efficiency worksheet (Worksheet 3)
The guide's positional slice is `iloc[4:14, :12]`. Because it is based on row positions seen in screenshots, the code below **checks that the extracted years are exactly 2016–2025** and stops (or falls back to locating year rows by value, with a warning) if they are not.

In [ ]:
EXPECTED_YEARS = list(range(2016, 2026))

def extract_annual_block(raw, row_slice, n_cols, names, label):
    """Positional extraction (as in the guide) + verification that years are exactly 2016-2025.
    If the positional slice is wrong, fall back to selecting rows whose first column is a year in 2016-2025.
    Raises an error if neither approach returns the expected 10 years."""
    block = raw.iloc[row_slice, :n_cols].copy()
    block.columns = names
    block["year"] = pd.to_numeric(block["year"], errors="coerce")
    years = block["year"].dropna().astype(int).tolist()
    if len(block) == len(EXPECTED_YEARS) and years == EXPECTED_YEARS:
        print(f"[{label}] positional extraction OK -> years {years[0]}-{years[-1]}")
        return block

    warnings.warn(f"[{label}] positional slice did NOT return 2016-2025 (got {years}). Trying year-based row detection.")
    cand = raw.iloc[:, :n_cols].copy()
    cand.columns = names
    cand["year"] = pd.to_numeric(cand["year"], errors="coerce")
    cand = cand[cand["year"].isin(EXPECTED_YEARS)].drop_duplicates("year", keep="first")
    if sorted(cand["year"].astype(int).tolist()) != EXPECTED_YEARS:
        raise ValueError(f"[{label}] Could not extract exactly 2016-2025. Inspect the raw worksheet before proceeding.")
    print(f"[{label}] year-based extraction used -> verify the values against the raw sheet!")
    return cand

wue_raw = pd.read_excel(water_accounts, sheet_name="3", header=None)
display(wue_raw.head(15))

In [ ]:
wue_names = ["year", "gva_agriculture", "gva_industry", "gva_services",
             "water_used_agriculture", "water_used_industry", "water_used_services",
             "wue_agriculture", "wue_industry", "wue_services", "wue_overall", "change_in_wue"]

wue = extract_annual_block(wue_raw, slice(4, 14), 12, wue_names, "WUE - Worksheet 3")
display(wue)
print("WUE records:", len(wue))
print("Years:", wue["year"].tolist())

## Step 4. Extract the Level of Water Stress worksheet (Worksheet 4)

In [ ]:
stress_raw = pd.read_excel(water_accounts, sheet_name="4", header=None)
display(stress_raw)

In [ ]:
stress_names = ["year", "water_withdrawal_mcm", "water_withdrawal_bcm", "water_stress"]

stress = extract_annual_block(stress_raw, slice(3, 13), 4, stress_names, "Water stress - Worksheet 4")
display(stress)
print("Water stress records:", len(stress))
print("Years:", stress["year"].tolist())

## Step 5. Verify numerical data in the Water Accounts worksheets

In [ ]:
print(wue.dtypes)
print(stress.dtypes)

In [ ]:
wue_na_before = wue.isnull().sum()
stress_na_before = stress.isnull().sum()

for col in wue.columns.drop("year"):
    wue[col] = pd.to_numeric(wue[col], errors="coerce")
for col in stress.columns.drop("year"):
    stress[col] = pd.to_numeric(stress[col], errors="coerce")

# Year as a clean integer key
wue["year"] = wue["year"].astype(int).astype("Int64")
stress["year"] = stress["year"].astype(int).astype("Int64")
wue = wue.reset_index(drop=True)
stress = stress.reset_index(drop=True)

# Investigate any NEW missing values created by the conversion
new_na_wue = (wue.isnull().sum() - wue_na_before)
new_na_stress = (stress.isnull().sum() - stress_na_before)
print("New NaNs created in WUE:\n", new_na_wue[new_na_wue != 0])
print("New NaNs created in Stress:\n", new_na_stress[new_na_stress != 0])

display(wue.describe())
display(stress.describe())
log_transform("Water Use Efficiency (WS 3)", "Annual rows extracted, columns named, converted to numeric", "Worksheet has title/header/notes rows")
log_transform("Water Stress (WS 4)", "Annual rows extracted, columns named, converted to numeric", "Worksheet has title/header/notes rows")

## Step 6. Verify measurement units

| Source | Unit |
|---|---|
| PSA Physical Supply / Use | million cubic meters (MCM) |
| PSA Water Use Efficiency (WS 3) | water in MCM; efficiency in PHP per m³ |
| PSA Water Stress (WS 4) | withdrawals in MCM and BCM; stress as a ratio |
| OWID Industrial water withdrawal | **must be confirmed from the OWID/FAO metadata** |

The OWID Philippine values are of the order of 10⁹–10¹⁰ (e.g. 9,008,000,374 for 2014), which is consistent with **cubic meters** per year. Confirm this on the OWID chart page ("Industrial water withdrawal", source FAO AQUASTAT) and then set the switch below. If it is not confirmed, leave it `False` – the conversion is then skipped.

In [ ]:
# >>> Set to True ONLY after confirming on OWID/FAO AQUASTAT metadata that the unit is cubic metres. <<<
OWID_UNIT_CONFIRMED_M3 = True

if OWID_UNIT_CONFIRMED_M3:
    withdrawal_clean["withdrawal_mcm"] = withdrawal_clean["industrial_water_withdrawal"] / 1_000_000
    log_transform("PHL industrial withdrawal", "withdrawal_mcm = industrial_water_withdrawal / 1,000,000",
                  "Convert m3 to million m3 so the unit matches PSA tables (after unit confirmation)")
    display(withdrawal_clean[["year", "industrial_water_withdrawal", "withdrawal_mcm"]].head())
else:
    print("Unit not confirmed -> no conversion performed.")

# Water stress: keep the ratio; add a percentage version for presentation only.
stress["water_stress_percentage"] = stress["water_stress"] * 100
display(stress)
log_transform("Water Stress (WS 4)", "water_stress_percentage = water_stress * 100", "Presentation only; original ratio retained")

## Step 7. Create selected annual indicators from the PSA supply and use tables

In [ ]:
print(supply_clean["inputs"].unique())
print(use_clean["inputs"].unique())

In [ ]:
supply_clean["input_label"] = (supply_clean["inputs"].astype(str).str.strip().str.replace(".", "", regex=False).str.upper())
use_clean["input_label"] = (use_clean["inputs"].astype(str).str.strip().str.replace(".", "", regex=False).str.upper())

print(supply_clean["input_label"].unique())
print(use_clean["input_label"].unique())

In [ ]:
# EXACT match on the verified labels (not a text search - 'TOTAL SUPPLY ABSTRACTED WATER' must NOT be captured).
supply_total = supply_clean[supply_clean["input_label"] == "TOTAL SUPPLY"][["year", "total_supply"]].copy()
use_total = use_clean[use_clean["input_label"] == "TOTAL USE"][["year", "total_use"]].copy()
supply_total = supply_total.sort_values("year").reset_index(drop=True)
use_total = use_total.sort_values("year").reset_index(drop=True)

print(supply_total)
print(use_total)
print("Supply total records:", len(supply_total), "| Use total records:", len(use_total))
assert supply_total["year"].is_unique and use_total["year"].is_unique
assert len(supply_total) == 16 and len(use_total) == 16, "Expected 16 annual totals (2010-2025)"
log_transform("PSA Supply / Use", "Extracted exact 'TOTAL SUPPLY' and 'TOTAL USE' rows (total_supply, total_use)",
              "National annual totals; aggregates and components kept distinct")

In [ ]:
# Sanity check: the supply and use totals should balance in a water-accounting framework.
chk = supply_total.merge(use_total, on="year")
chk["difference"] = chk["total_supply"] - chk["total_use"]
display(chk)

> **Not done on purpose:** an industry-only series from the PSA tables. The combined industry column (*Mining and Quarrying, Manufacturing, and Construction*) and `TOTAL INDUSTRY` must only be used after choosing the correct accounting row (e.g. abstraction vs. own-use vs. distributed water) from the PSA definitions. The full cleaned tables (`supply_clean`, `use_clean`) are saved so that this selection can be made later without re-processing.

---
# 4.4 DATA INTEGRATION
## Step 1. Verify the Year columns

In [ ]:
for name, df in {
    "Supply Total": supply_total,
    "Use Total": use_total,
    "Withdrawal": withdrawal_clean,
    "WUE": wue,
    "Stress": stress,
}.items():
    print("\n", name)
    print(df["year"].min(), "-", df["year"].max())
    print("Duplicate years:", df["year"].duplicated().sum())

## Step 2. Check the reporting periods

| Dataset | Reporting period |
|---|---|
| Physical Water Supply | 2010–2025 |
| Physical Water Use | 2010–2025 |
| Industrial Water Withdrawal (PHL) | 1995–2023 |
| Water Use Efficiency | 2016–2025 |
| Level of Water Stress | 2016–2025 |

The common period is **2016–2023**. The longer individual datasets are retained for analyses that don't need every indicator.

In [ ]:
periods = {
    "Supply Total": supply_total, "Use Total": use_total, "PHL Withdrawal": withdrawal_clean,
    "WUE": wue, "Stress": stress,
}
common_years = set.intersection(*[set(df["year"].dropna().astype(int)) for df in periods.values()])
print("Common years across all five annual datasets:", min(common_years), "-", max(common_years), sorted(common_years))
assert sorted(common_years) == list(range(2016, 2024))

## Step 3. Merge the annual supply and use totals

In [ ]:
annual_supply_use = pd.merge(supply_total, use_total, on="year", how="outer")
annual_supply_use = annual_supply_use.sort_values("year")
display(annual_supply_use)

## Step 4. Merge industrial water withdrawal (OWID – kept as its own, clearly named variable)

In [ ]:
if "withdrawal_mcm" in withdrawal_clean.columns:
    withdrawal_annual = withdrawal_clean[["year", "withdrawal_mcm"]].rename(
        columns={"withdrawal_mcm": "owid_industrial_withdrawal_mcm"}).copy()
else:
    withdrawal_annual = withdrawal_clean[["year", "industrial_water_withdrawal"]].rename(
        columns={"industrial_water_withdrawal": "owid_industrial_withdrawal_raw_unit_unconfirmed"}).copy()
display(withdrawal_annual.head())

## Step 5. Prepare the WUE and water-stress indicators

In [ ]:
wue_selected = wue[["year", "water_used_industry", "wue_industry", "wue_overall"]].copy()
stress_selected = stress[["year", "water_withdrawal_mcm", "water_stress"]].copy()
display(wue_selected)
display(stress_selected)

## Step 6. Integrate the annual datasets (outer joins – missing years are *not* filled)

In [ ]:
annual_master = annual_supply_use.merge(withdrawal_annual, on="year", how="outer")
annual_master = annual_master.merge(wue_selected, on="year", how="outer")
annual_master = annual_master.merge(stress_selected, on="year", how="outer")
annual_master = annual_master.sort_values("year").reset_index(drop=True)
display(annual_master)
print("Annual master:", annual_master.shape, "| years", annual_master["year"].min(), "-", annual_master["year"].max())
log_transform("Annual master", "Outer-joined supply/use totals, OWID withdrawal, WUE and water stress on year",
              "Preserve all years; gaps indicate a source has no observation for that year")

## Step 7. Create the common-period analytical dataset (2016–2023)

In [ ]:
common_period = annual_master[(annual_master["year"] >= 2016) & (annual_master["year"] <= 2023)].copy()
display(common_period)

print("Common-period dimensions:", common_period.shape)
print("\nMissing values:")
display(common_period.isnull().sum())

## Step 8. Check the integrated data

In [ ]:
print("Duplicate years:")
print(common_period["year"].duplicated().sum())
print("\nData types:")
print(common_period.dtypes)
print("\nNumerical summary:")
display(common_period.describe())

## Step 9. Integration limitations (read before analysing)
- OWID *Industrial Water Withdrawal* = self-supplied industrial withdrawals (FAO AQUASTAT definition).
- PSA *Physical Water Use* follows the water-accounting framework with different categories.
- PSA *industry* = mining, manufacturing and construction combined (not mining-only).
- *Water Use Efficiency* is a separate indicator (PHP per m³) – not industrial withdrawal.
- *Water stress* = national freshwater withdrawal relative to renewable freshwater resources and environmental-flow requirements.
- The merged table supports exploring **patterns and possible relationships only** – no causal claims.

---
# 4.5 PREPARED DATASET DESCRIPTION AND VALIDATION
## Step 1. Inspect the final datasets

In [ ]:
prepared_datasets = {
    "Supply Total": supply_total,
    "Use Total": use_total,
    "Philippine Industrial Withdrawal": withdrawal_clean,
    "Water Use Efficiency": wue,
    "Water Stress": stress,
    "Annual Master": annual_master,
    "Common Period": common_period,
}

for name, df in prepared_datasets.items():
    print("\n" + "=" * 50)
    print("DATASET:", name)
    print("=" * 50)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nDuplicate rows:")
    print(df.duplicated().sum())
    print("\nData types:")
    print(df.dtypes)

## Step 2. Validate the Year ranges

In [ ]:
for name, df in prepared_datasets.items():
    if "year" in df.columns:
        print("\nDataset:", name)
        print("Minimum year:", df["year"].min())
        print("Maximum year:", df["year"].max())
        print("Unique years:", df["year"].nunique())

## Step 3. Compare the prepared datasets with the original datasets

In [ ]:
print("Original withdrawal records:", len(withdrawal))
print("Philippine withdrawal records:", len(withdrawal_clean))
print("Original supply dimensions:", supply.shape)
print("Prepared supply dimensions:", supply_clean.shape)
print("Original use dimensions:", use.shape)
print("Prepared use dimensions:", use_clean.shape)

**Explaining the differences:** the withdrawal file was reduced from all countries/regions to the 29 PHL rows (justified selection). Supply and use keep the same number of rows; the only column change is the helper column `input_label` (so the prepared tables have one extra column).

In [ ]:
# Automatic validation checks
checks = {
    "Originals unchanged": supply.equals(supply_work) and use.equals(use_work) and withdrawal.equals(withdrawal_work),
    "PHL subset only": (withdrawal_clean["code"] == "PHL").all(),
    "PHL subset has 29 rows (1995-2023)": len(withdrawal_clean) == 29,
    "Supply/Use totals 2010-2025 (16 years)": len(supply_total) == 16 and len(use_total) == 16,
    "WUE years 2016-2025": wue["year"].astype(int).tolist() == list(range(2016, 2026)),
    "Stress years 2016-2025": stress["year"].astype(int).tolist() == list(range(2016, 2026)),
    "No duplicate years in annual master": not annual_master["year"].duplicated().any(),
    "Common period = 2016-2023 (8 rows)": common_period["year"].astype(int).tolist() == list(range(2016, 2024)),
    "No analytical NaNs in common period": int(common_period.isnull().sum().sum()) == 0,
}
for k, v in checks.items():
    print(("✔" if v else "✘"), k)

## Step 4. Save the prepared datasets

In [ ]:
os.makedirs("Prepared_Datasets", exist_ok=True)

supply_clean.to_csv("Prepared_Datasets/physical_water_supply_clean.csv", index=False)
use_clean.to_csv("Prepared_Datasets/physical_water_use_clean.csv", index=False)
withdrawal_clean.to_csv("Prepared_Datasets/philippine_industrial_withdrawal_clean.csv", index=False)
wue.to_csv("Prepared_Datasets/water_use_efficiency_clean.csv", index=False)
stress.to_csv("Prepared_Datasets/water_stress_clean.csv", index=False)
annual_master.to_csv("Prepared_Datasets/annual_master_dataset.csv", index=False)
common_period.to_csv("Prepared_Datasets/common_period_dataset.csv", index=False)

# Extra: the annual totals and the change records requested in the guide's hand-back note
supply_total.to_csv("Prepared_Datasets/supply_total_annual.csv", index=False)
use_total.to_csv("Prepared_Datasets/use_total_annual.csv", index=False)
pd.DataFrame(cleaning_log).to_csv("Prepared_Datasets/cleaning_log.csv", index=False)
pd.DataFrame(transformation_log).to_csv("Prepared_Datasets/transformation_log.csv", index=False)

print(sorted(os.listdir("Prepared_Datasets")))

## Step 5. Download the prepared datasets

In [ ]:
shutil.make_archive("Prepared_Datasets", "zip", "Prepared_Datasets")

if IN_COLAB:
    from google.colab import files
    files.download("Prepared_Datasets.zip")
    # Single file example:
    # files.download("Prepared_Datasets/common_period_dataset.csv")
else:
    print("Not running in Colab - find Prepared_Datasets.zip in the working directory.")

---
# Next: Orange Data Mining
Load the CSVs from `Prepared_Datasets/` (not the originals) into Orange: **File → Data Table → Select Columns → Feature Statistics → Distributions → Scatter Plot**. Don't use *Impute* unless a missing value has been shown to be a genuine gap. Save the workflow (`.ows`) with screenshots.

# Final validation checklist
- [ ] Original datasets preserved · [ ] Working copies used · [ ] PHL records selected
- [ ] Accounting categories identified · [ ] Missing values investigated before treatment · [ ] Duplicates checked
- [ ] Zeros / unusual values investigated · [ ] Water Accounts annual records extracted (2016–2025)
- [ ] Units verified · [ ] Year used as merge key · [ ] OWID vs PSA definitions kept distinct
- [ ] CSVs exported · [ ] Orange workflow saved

**Reminders:** don't fill all NaN with 0 · don't delete zeros/outliers without justification · withdrawal ≠ consumption · PSA industry ≠ mining-only · correlation ≠ causation.